<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum04/BD_A_P04_2411531001_Habibi_Putra_Rizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRAKTIKUM BIG DATA 4
## NoSQL dan Pemodelan Data

**Nama:** Habibi Putra Rizqullah  
**NIM:** 2411531001  
**Kelas:** A  


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## K-1 Menyiapkan Lingkungan dan Data Contoh

### Tujuan
Memasang pustaka, menghubungkan Google Drive, membaca `trips_bersih` hasil Praktikum 3, dan menyederhanakan nama kolom.

### Prediksi sebelum eksperimen
Folder `trips_bersih` diperkirakan dapat dibaca langsung dari Google Drive. Waktu pembacaan diperkirakan berada pada hitungan detik karena data berisi jutaan baris.

In [ ]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import zipfile

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH_P4 = f"{DIR_SIMPAN}/trips_bersih"
TRIPS_BERSIH_P3 = f"{DIR_P3}/trips_bersih"
ZIP_TRIPS = f"{DIR_P3}/trips_bersih.zip"

os.makedirs(DIR_SIMPAN, exist_ok=True)

if os.path.isdir(TRIPS_BERSIH_P4):
    TRIPS_BERSIH = TRIPS_BERSIH_P4
elif os.path.isdir(TRIPS_BERSIH_P3):
    TRIPS_BERSIH = TRIPS_BERSIH_P3
else:
    TRIPS_BERSIH = TRIPS_BERSIH_P3

if not os.path.exists(TRIPS_BERSIH) and os.path.exists(ZIP_TRIPS):
    HASIL_EKSTRAK = "/content/trips_bersih_p04"
    os.makedirs(HASIL_EKSTRAK, exist_ok=True)
    with zipfile.ZipFile(ZIP_TRIPS, "r") as arsip:
        arsip.extractall(HASIL_EKSTRAK)

    kandidat = [
        os.path.join(HASIL_EKSTRAK, "trips_bersih"),
        HASIL_EKSTRAK,
    ]
    TRIPS_BERSIH = next(
        (p for p in kandidat if any(f.endswith(".parquet") for _, _, fs in os.walk(p) for f in fs)),
        TRIPS_BERSIH,
    )
    print("Dataset diekstrak dari ZIP:", TRIPS_BERSIH)

assert os.path.exists(TRIPS_BERSIH), (
    "Folder trips_bersih tidak ada. Jalankan ulang P3/K-10 atau gunakan K-1b."
)

print("Folder Praktikum 3:", DIR_P3)
print("Folder Praktikum 4:", DIR_SIMPAN)
print("Dataset:", TRIPS_BERSIH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset diekstrak dari ZIP: /content/trips_bersih_p04
Folder Praktikum 3: /content/drive/MyDrive/BigData/Praktikum3
Folder Praktikum 4: /content/drive/MyDrive/BigData/Praktikum4
Dataset: /content/trips_bersih_p04


In [ ]:
import time
import json
import sqlite3
import random
import shutil
import glob

import numpy as np
import pandas as pd

catatan = []

def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

from urllib.parse import unquote

def baca_trips_berpartisi(folder):
    """Membaca file Parquet per partisi untuk menghindari ArrowInvalid.

    PyArrow kadang gagal menyatukan dictionary partisi apabila terdapat
    nilai null. Nama borough dipulihkan dari nama folder borough_naik=... .
    """
    daftar_file = sorted(
        glob.glob(os.path.join(folder, "**", "*.parquet"), recursive=True)
    )
    if not daftar_file:
        raise FileNotFoundError(f"Tidak ada file Parquet di dalam {folder}")

    bagian = []
    for path_file in daftar_file:
        potongan = pd.read_parquet(path_file)

        if "borough_naik" not in potongan.columns:
            nilai_partisi = None
            for nama_folder in path_file.split(os.sep):
                if nama_folder.startswith("borough_naik="):
                    nilai_partisi = unquote(nama_folder.split("=", 1)[1])
                    break
            potongan["borough_naik"] = (
                None if nilai_partisi == "__HIVE_DEFAULT_PARTITION__" else nilai_partisi
            )

        potongan["borough_naik"] = (
            potongan["borough_naik"].astype("string").fillna("Unknown")
        )
        bagian.append(potongan)

    return pd.concat(bagian, ignore_index=True)

df = ukur(
    "baca trips_bersih",
    lambda: baca_trips_berpartisi(TRIPS_BERSIH),
)

NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}

df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

print("jumlah baris sampel:", f"{len(sample):,}")
print("kolom:", sample.columns.tolist())
display(sample.head())

[baca trips_bersih] 2.7022 s
baris: 2,986,910 | kolom: 18
jumlah baris sampel: 300,000
kolom: ['waktu_jemput', 'tpep_dropoff_datetime', 'passenger_count', 'jarak_mil', 'zona_jemput', 'zona_tujuan', 'payment_type', 'fare_amount', 'tip', 'total_bayar', 'durasi_menit', 'jam', 'metode_bayar', 'jam_mulai', 'suhu_c', 'hujan_mm', 'tanggal', 'wilayah_jemput', 'id_perjalanan']


,waktu_jemput,tpep_dropoff_datetime,passenger_count,jarak_mil,zona_jemput,zona_tujuan,payment_type,fare_amount,tip,total_bayar,durasi_menit,jam,metode_bayar,jam_mulai,suhu_c,hujan_mm,tanggal,wilayah_jemput,id_perjalanan
0,2023-01-16 10:40:39,2023-01-16 10:52:29,1.0,1.55,48,164,1,12.1,3.22,19.32,11.833333,10,Kartu kredit,2023-01-16 10:00:00,0.5,0.0,2023-01-16,Manhattan,1
1,2023-01-16 18:09:53,2023-01-16 18:16:40,2.0,0.57,164,170,1,7.2,2.24,13.44,6.783333,18,Kartu kredit,2023-01-16 18:00:00,3.2,0.0,2023-01-16,Manhattan,2
2,2023-01-18 15:29:58,2023-01-18 15:44:02,1.0,1.80,234,230,1,12.1,3.20,19.30,14.066667,15,Kartu kredit,2023-01-18 15:00:00,10.5,0.0,2023-01-18,Manhattan,3
3,2023-01-10 15:52:40,2023-01-10 16:14:36,1.0,3.04,48,114,1,21.2,5.04,30.24,21.933333,15,Kartu kredit,2023-01-10 15:00:00,6.3,0.0,2023-01-10,Manhattan,4
4,2023-01-24 14:11:55,2023-01-24 14:35:33,1.0,3.90,107,143,1,24.0,5.00,33.00,23.633333,14,Kartu kredit,2023-01-24 14:00:00,7.8,0.0,2023-01-24,Manhattan,5


## K-2 Tabel Biasa vs Key-value

### Prediksi sebelum eksperimen
Pencarian berdasarkan ID diperkirakan paling cepat pada key-value. Tabel SQL berindeks diperkirakan mendekati key-value, sedangkan SQL tanpa indeks diperkirakan jauh lebih lambat. Untuk agregasi seluruh data, SQL `GROUP BY` diperkirakan lebih cepat daripada membuka seluruh nilai JSON.

In [ ]:
KOLOM = [
    "id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
    "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"
]
data = sample[KOLOM].copy()

if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

def bangun_kv():
    return {
        f"perjalanan:{r['id_perjalanan']}": json.dumps(r)
        for r in data.to_dict("records")
    }

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

[relasional: muat tabel] 2.5168 s
[key-value: bangun dict] 9.6995 s
jumlah key: 300000


In [ ]:
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [
        cur.execute(
            "SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)
        ).fetchone()
        for i in id_uji
    ]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)

print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

[cari 200 perjalanan - SQL tanpa indeks] 4.3215 s
[cari 200 perjalanan - SQL dengan indeks] 0.0037 s
[cari 200 perjalanan - key-value] 0.0012 s
hasil sama? True


In [ ]:
def hitung_sql():
    return con.execute("""
        SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
        FROM perjalanan
        GROUP BY wilayah_jemput
        ORDER BY 2 DESC, 1
    """).fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted(
        [(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
        key=lambda x: (-x[1], x[0]),
    )

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[hitung per wilayah - SQL GROUP BY] 0.1951 s
[hitung per wilayah - key-value (buka semua nilai)] 1.4229 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3824, 39.42)]
hasil sama? True


### Pembahasan K-2

Dari hasil percobaan, mencari 200 id lewat SQL tanpa indeks memakan 4,3215 detik, sedangkan setelah indeks dibuat hanya 0,0037 detik. Selisihnya sekitar 1.170 kali. Wajar, karena tanpa indeks SQLite harus mengecek baris satu per satu sampai ketemu, sementara dengan indeks letak barisnya langsung diketahui.

Key-value masih lebih cepat dari tabel berindeks, 0,0012 detik lawan 0,0037 detik (sekitar 3,1 kali). Tapi selisih sebenarnya cuma 0,0025 detik untuk 200 pencarian, jadi dalam pemakaian biasa tidak akan terasa. Ini sesuai prediksi bahwa tabel berindeks mendekati key-value. Keunggulan key-value yang sebenarnya ada di hal lain: kuncinya langsung menunjuk letak data sehingga mudah dibagi ke banyak mesin, dan itu tidak kelihatan pada satu komputer.

Untuk menghitung rata-rata per wilayah, key-value justru kalah, yaitu 1,4229 detik lawan 0,1951 detik pada SQL GROUP BY, atau sekitar 7,3 kali lebih lambat. Penyebabnya, nilai di key-value hanya berupa teks JSON, jadi 300.000 nilai harus dibuka satu per satu dengan json.loads sebelum bisa dijumlahkan. SQL sudah tahu bahwa total_bayar adalah kolom angka.

## K-3 Document: Bentuk Data yang Fleksibel

### Prediksi sebelum eksperimen
Model document diperkirakan dapat menerima field baru tanpa perubahan skema. Kemudahan tersebut juga diperkirakan meningkatkan risiko ketidakkonsistenan nama atau tipe field.

In [ ]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {
            "zona": int(r.zona_jemput),
            "wilayah": r.wilayah_jemput,
        },
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {
            "jarak_mil": float(r.jarak_mil),
            "total": float(r.total_bayar),
            "metode_bayar": r.metode_bayar,
        },
    }
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)

print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}


In [ ]:
Q = Query()

print(
    "jemput di Manhattan & total > 50 :",
    len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))),
)
print("dokumen yang punya field tip     :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip     :", len(tabel.search(~Q.biaya.tip.exists())))

tabel.insert({
    "id_perjalanan": 999_999,
    "waktu_jemput": "2023-01-15 08:00:00",
    "jemput": {"zona": 161, "wilayah": "Manhattan"},
    "tujuan": {"zona": 236},
    "biaya": {
        "jarak_mil": 1.2,
        "total": 14.5,
        "metode_bayar": "Kartu kredit",
        "tip": 3.0,
    },
    "cuaca": {"hujan": True, "suhu_c": -2.0},
})

print("dokumen yang punya field cuaca   :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jemput di Manhattan & total > 50 : 693
dokumen yang punya field tip     : 15906
dokumen yang TIDAK punya tip     : 4094
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


### Pembahasan K-3

Dokumen ke-20.001 bisa langsung membawa field baru, yaitu cuaca, tanpa ALTER TABLE atau migrasi apa pun. Ada juga 4.094 dari 20.000 dokumen yang memang tidak punya field tip, dan di tabel relasional itu akan menjadi kolom NULL. Keuntungannya ada di fleksibilitas bentuk data. Risikonya, kalau ada salah ketik nama field, misalnya biya, database tidak akan menegur karena tidak ada skema yang dilanggar. Jadi validasi data sepenuhnya menjadi urusan aplikasi.

## K-4 Penyimpanan per Kolom: DuckDB pada Parquet

### Prediksi sebelum eksperimen
DuckDB diperkirakan lebih cepat daripada SQLite untuk agregasi karena hanya membaca kolom yang diperlukan dan dapat menjalankan pemrosesan vektor. Perkiraan awal: DuckDB sekitar **5–15 kali** lebih cepat untuk query agregasi ini.

In [ ]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
if not glob.glob(GLOB):
    GLOB = f"{TRIPS_BERSIH}/**/*.parquet"

con_d = duckdb.connect()
con_d.sql(f"""
    CREATE OR REPLACE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID         AS zona_jemput,
           DOLocationID         AS zona_tujuan,
           CASE WHEN borough_naik IS NULL
                OR borough_naik = '__HIVE_DEFAULT_PARTITION__'
                THEN 'Unknown' ELSE borough_naik END AS wilayah_jemput,
           trip_distance        AS jarak_mil,
           total_amount         AS total_bayar,
           nama_pembayaran      AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)
""")

SQL = """
    SELECT wilayah_jemput,
           COUNT(*) AS jumlah,
           ROUND(AVG(total_bayar), 2) AS rata_bayar
    FROM perjalanan
    GROUP BY wilayah_jemput
    ORDER BY jumlah DESC, wilayah_jemput
"""

hasil_d = ukur(
    "DuckDB - hitung per wilayah (langsung dari Parquet)",
    lambda: con_d.sql(SQL).df(),
)
display(hasil_d)

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.1139 s


,wilayah_jemput,jumlah,rata_bayar
0,Manhattan,2659609,22.72
1,Queens,270315,70.46
2,Unknown,37949,39.56
3,Brooklyn,15724,32.72
4,Bronx,3074,34.44
5,Staten Island,211,74.07
6,EWR,28,106.41


In [ ]:
KOLOM_S = [
    "wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"
]

if os.path.exists("baris.db"):
    os.remove("baris.db")

con_s = sqlite3.connect("baris.db")
ukur(
    "SQLite - muat seluruh baris ke tabel",
    lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False),
)

hasil_s = ukur(
    "SQLite - hitung per wilayah",
    lambda: con_s.execute("""
        SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
        FROM perjalanan
        GROUP BY wilayah_jemput
        ORDER BY 2 DESC, 1
    """).fetchall(),
)

hasil_duckdb = [tuple(r) for r in hasil_d.itertuples(index=False)]
hasil_sqlite = [tuple(r) for r in hasil_s]
print("hasil sama?", hasil_duckdb == hasil_sqlite)

[SQLite - muat seluruh baris ke tabel] 9.1240 s
[SQLite - hitung per wilayah] 1.7269 s
hasil sama? True


In [ ]:
SQL2 = """
    SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
    FROM perjalanan
    WHERE wilayah_jemput = 'Staten Island'
"""

print(
    ukur(
        "DuckDB - hanya wilayah Staten Island",
        lambda: con_d.sql(SQL2).fetchall(),
    )
)

ukuran_pq = sum(
    os.path.getsize(os.path.join(root, nama))
    for root, _, files in os.walk(TRIPS_BERSIH)
    for nama in files
    if nama.endswith(".parquet")
) / 1024**2

ukuran_sqlite = os.path.getsize("baris.db") / 1024**2
print(
    f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | "
    f"SQLite ({len(KOLOM_S)} kolom): {ukuran_sqlite:.1f} MB"
)

[DuckDB - hanya wilayah Staten Island] 0.0176 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


### Pembahasan K-4

DuckDB menghitung rata-rata per wilayah langsung dari Parquet dalam 0,1139 detik, sedangkan SQLite butuh 1,7269 detik, jadi sekitar 15 kali lebih cepat. Angka itu ada di batas atas prediksi (5 sampai 15 kali) dan belum termasuk waktu memuat data ke tabel SQLite yang sendiri memakan 9,1240 detik, sementara DuckDB tidak perlu tahap itu.

Ada dua alasan di balik selisih itu. Pertama, query hanya perlu membaca 2 dari 18 kolom (column pruning). Kedua, DuckDB memproses banyak baris sekaligus dan memakai beberapa inti prosesor, dan hal ini tidak ada kaitannya dengan format kolom. Jadi tidak semua keunggulannya berasal dari Parquet.

Ketika query dibatasi ke wilayah Staten Island, waktunya turun menjadi 0,0176 detik, sekitar 6,5 kali lebih cepat dari hitungan penuh, karena DuckDB hanya membuka folder partisi wilayah itu. Dari sisi ukuran, Parquet dengan 18 kolom (66,5 MB) justru lebih kecil dari tabel SQLite yang hanya 5 kolom (114,2 MB). Hasil kedua mesin sudah sama (`hasil sama? True`) setelah label wilayah yang tidak diketahui disamakan menjadi Unknown.

## K-5 Graph: Pertanyaan tentang Keterhubungan

### Prediksi sebelum eksperimen
Graph dan SQL diperkirakan memberikan hasil yang sama untuk dua langkah. Pada data dengan jumlah zona terbatas, selisih waktunya mungkin kecil. Graph diperkirakan lebih mudah dikembangkan ketika jumlah langkah berubah.

In [ ]:
import networkx as nx

arus = ukur(
    "hitung arus antar zona",
    lambda: con_d.sql("""
        SELECT zona_jemput AS asal,
               zona_tujuan AS tujuan,
               COUNT(*) AS jumlah
        FROM perjalanan
        GROUP BY 1, 2
    """).df(),
)

G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))

print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

top_graf = sorted(
    G.edges(data=True),
    key=lambda e: (-e[2]["jumlah"], e[0], e[1]),
)[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])

top_sql = arus.sort_values(
    ["jumlah", "asal", "tujuan"], ascending=[False, True, True]
).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

populer = sorted(
    G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0])
)[:5]
print("zona tujuan terpopuler:", populer)

[hitung arus antar zona] 0.2170 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL  : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]


In [ ]:
MIN = 50
zona_awal = populer[0][0]

def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

con_d.register("arus_df", arus)

def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus_df a1
        JOIN arus_df a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal}
          AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN}
          AND a2.tujuan <> {zona_awal}
    """).df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)

print(
    "zona awal:", zona_awal,
    "| terjangkau (graph):", len(hg),
    "| (SQL):", len(hs),
    "| sama?", hg == hs,
)

H = nx.DiGraph([
    (a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN
])

for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[2 langkah - graph (NetworkX)] 0.0068 s
[2 langkah - SQL self-join (DuckDB)] 0.0081 s
zona awal: 236 | terjangkau (graph): 214 | (SQL): 214 | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


### Pembahasan K-5

Untuk mencari zona yang bisa dicapai dalam dua langkah dari zona 236, graph butuh 0,0068 detik dan SQL self-join butuh 0,0081 detik. Graph sedikit lebih cepat, tetapi selisihnya hanya sekitar 1,3 milidetik pada 262 zona, jadi menurut saya angka itu belum cukup untuk menyimpulkan graph selalu lebih cepat. Hasil keduanya sama, yaitu 214 zona, dan lima arus terbesar dari graph juga identik dengan hasil SQL.

Perbedaan yang lebih terasa ada pada cara menulis pertanyaannya. Di SQL, dua langkah berarti satu self-join, tiga langkah berarti dua self-join, dan seterusnya. Di graph cukup mengganti angka k. Hasil untuk k = 1 sampai 4 berturut-turut 74, 214, 214, dan 214 zona. Artinya sejak langkah kedua jangkauannya sudah penuh, dan menambah langkah tidak menemukan zona baru.

## K-6 Indeks Spasial

### Prediksi sebelum eksperimen
STRtree diperkirakan jauh lebih cepat dibanding loop Python karena tidak memeriksa seluruh titik satu per satu. Dibanding NumPy, selisihnya diperkirakan lebih kecil karena NumPy sudah melakukan operasi secara vektor.

**Catatan:** titik koordinat pada langkah ini bersifat sintetis karena data TLC yang digunakan hanya menyimpan ID zona, bukan koordinat perjalanan.

In [ ]:
from shapely import STRtree, box, points

rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)
titik = points(bujur, lintang)

M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [
    box(x - 0.005, y - 0.005, x + 0.005, y + 0.005)
    for x, y in zip(cx, cy)
]

def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        mask = (
            (bujur >= x - 0.005)
            & (bujur <= x + 0.005)
            & (lintang >= y - 0.005)
            & (lintang <= y + 0.005)
        )
        hasil.append(int(mask.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur(
    "cara 3 - STRtree, 1000 kotak",
    lambda: [len(pohon.query(k, predicate="contains")) for k in kotak],
)

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx)
print("loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 - loop Python, 5 kotak] 13.9702 s
[cara 2 - NumPy, 1000 kotak] 0.5210 s
[bangun indeks STRtree] 0.0788 s
[cara 3 - STRtree, 1000 kotak] 0.2139 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True
loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


### Pembahasan K-6

Loop Python butuh 13,9702 detik untuk 5 kotak, atau sekitar 2,79 detik per kotak. Untuk 1.000 kotak perkiraannya sekitar 2.794 detik (kurang lebih 47 menit). Angka ini hanya perkiraan dan tidak dijalankan. STRtree menyelesaikan 1.000 kotak dalam 0,2927 detik, yaitu 0,0788 detik untuk membangun indeks dan 0,2139 detik untuk mencari. Dibanding perkiraan loop, itu sekitar 9.500 kali lebih cepat.

Pembanding yang lebih adil adalah NumPy, yang butuh 0,5210 detik. Terhadap NumPy, STRtree hanya sekitar 1,8 kali lebih cepat jika waktu membangun indeks dihitung, dan sekitar 2,4 kali jika tidak. Dua angka itu beda jauh karena loop Python memeriksa titik satu per satu lewat objek Shapely, sedangkan NumPy sudah membandingkan semua titik sekaligus. Jadi kalimat "indeks sekian kali lebih cepat" tidak ada artinya kalau pembandingnya tidak disebut. Perlu diingat juga bahwa titiknya sintetis, bukan data perjalanan sebenarnya.

## K-7 Delta Lake: Versi, Time Travel, dan Schema Enforcement

### Prediksi sebelum eksperimen
Setiap penulisan dan koreksi diperkirakan menghasilkan versi baru. Versi lama diperkirakan tetap dapat dibaca, sedangkan penambahan kolom asing diperkirakan ditolak tanpa mengubah nomor versi.

In [ ]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = [
    "waktu_jemput", "zona_jemput", "zona_tujuan",
    "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"
]

data_delta = df[KOLOM_DELTA].sample(
    n=min(200_000, len(df)), random_state=42
).reset_index(drop=True)

PATH_DELTA = os.path.join(os.getcwd(), "trips_delta")
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

sekarang = data_delta.iloc[:150_000]
n_koreksi = int((
    (sekarang["wilayah_jemput"] == "Bronx")
    & (sekarang["metode_bayar"] == "Tunai")
).sum())
print("baris yang akan dikoreksi:", n_koreksi)

dt.update(
    predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
    updates={"total_bayar": "total_bayar + 1.0"},
)
print("versi:", DeltaTable(PATH_DELTA).version())

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2


In [ ]:
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()

print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print(
    "selisih total_bayar (terbaru - v1):",
    round(selisih, 2),
    "| jumlah baris dikoreksi:",
    n_koreksi,
)

print("\nRiwayat perubahan:")
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

print("\nIsi _delta_log:")
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0 | jumlah baris dikoreksi: 20

Riwayat perubahan:
2 UPDATE
1 WRITE
0 WRITE

Isi _delta_log:
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']


In [ ]:
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1

try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)

print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


### Pembahasan K-7

Setiap perubahan tabel menghasilkan versi baru dan semuanya masih bisa dibuka. Versi 0 berisi 100.000 baris, versi 1 berisi 150.000 baris, dan versi 2 (setelah update) tetap 150.000 baris. Selisih total_bayar antara versi terbaru dan versi 1 adalah 20,0, sama persis dengan 20 baris Bronx berpembayaran Tunai yang dikoreksi sebesar 1,0 per baris. Jadi update hanya mengubah baris yang dituju dan data versi lama tidak hilang.

Percobaan menambah data dengan kolom_asing ditolak dengan SchemaMismatchError dan nomor versi tetap 2, artinya penulisan yang gagal tidak meninggalkan data setengah jadi. Riwayat tabel mencatat WRITE, WRITE, UPDATE, dan folder _delta_log berisi tiga berkas JSON, satu untuk tiap versi.

## K-8 Small File Problem pada Tabel Delta

### Prediksi sebelum eksperimen
Dua puluh append kecil diperkirakan menambah jumlah file aktif. Compaction diperkirakan mengurangi file aktif tanpa mengubah jumlah baris, tetapi file lama masih berada di disk untuk mendukung time travel.

In [ ]:
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

for i in range(20):
    awal = 150_000 + i * 500
    akhir = 150_000 + (i + 1) * 500
    potongan = data_delta.iloc[awal:akhir]
    write_deltalake(PATH_DELTA, potongan, mode="append")

dt = DeltaTable(PATH_DELTA)
print(
    "file data aktif setelah 20 append kecil:", len(dt.file_uris()),
    "| versi:", dt.version(),
    "| baris:", len(dt.to_pandas()),
)

t0 = time.perf_counter()
_ = DeltaTable(PATH_DELTA).to_pandas()
t_sebelum = time.perf_counter() - t0

hasil = dt.optimize.compact()
print(
    "hasil compact:",
    {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")},
)

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter()
_ = DeltaTable(PATH_DELTA).to_pandas()
t_sesudah = time.perf_counter() - t0

print(
    "file data aktif setelah compact:", len(dt.file_uris()),
    "| versi:", dt.version(),
    "| baris:", len(dt.to_pandas()),
)
print(
    f"waktu baca sebelum compact: {t_sebelum:.4f} s | "
    f"sesudah: {t_sesudah:.4f} s"
)

print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(
    retention_hours=0,
    enforce_retention_duration=False,
    dry_run=True,
)
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0780 s | sesudah: 0.0525 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


### Pembahasan K-8

Sebelum append, file aktif ada 1. Setelah 20 kali append kecil naik menjadi 21 (versi 22), lalu kembali menjadi 1 setelah compaction (versi 23). Compaction menambah 1 file dan mengeluarkan 21 file dari tabel, sedangkan jumlah barisnya tetap 160.000. Waktu baca turun dari 0,0780 detik menjadi 0,0525 detik, sekitar 1,5 kali. Selisihnya kecil karena datanya cuma 160.000 baris. Pada tabel besar dengan ribuan file kecil, bedanya akan lebih besar.

Meski file aktif tinggal 1, di disk masih ada 24 file Parquet. File lama sengaja tidak dihapus karena masih dibutuhkan untuk time travel ke versi sebelumnya. VACUUM dengan dry_run=True hanya melaporkan 23 file yang bisa dibersihkan dan tidak menghapus apa pun. VACUUM dengan retention 0 jam tidak dijalankan sungguhan karena akan memutus time travel.

## K-9 Merangkum dan Memilih

### Tujuan
Menyimpan seluruh catatan waktu ke `benchmark_nosql.csv` dan merangkum kecocokan tiap model penyimpanan.

In [ ]:
rekap = pd.DataFrame(catatan)
PATH_BENCHMARK = f"{DIR_SIMPAN}/benchmark_nosql.csv"
rekap.to_csv(PATH_BENCHMARK, index=False)

print(rekap.to_string(index=False))
print("\nBenchmark tersimpan di:", PATH_BENCHMARK)

                                            langkah   detik
                                  baca trips_bersih  2.7022
                             relasional: muat tabel  2.5168
                             key-value: bangun dict  9.6995
             cari 200 perjalanan - SQL tanpa indeks  4.3215
            cari 200 perjalanan - SQL dengan indeks  0.0037
                    cari 200 perjalanan - key-value  0.0012
                  hitung per wilayah - SQL GROUP BY  0.1951
  hitung per wilayah - key-value (buka semua nilai)  1.4229
DuckDB - hitung per wilayah (langsung dari Parquet)  0.1139
               SQLite - muat seluruh baris ke tabel  9.1240
                        SQLite - hitung per wilayah  1.7269
               DuckDB - hanya wilayah Staten Island  0.0176
                             hitung arus antar zona  0.2170
                       2 langkah - graph (NetworkX)  0.0068
                 2 langkah - SQL self-join (DuckDB)  0.0081
                      cara 1 - loop Pyth

In [ ]:
waktu = rekap.set_index("langkah")["detik"]

def rasio(pembilang, penyebut):
    return waktu[pembilang] / waktu[penyebut]

ringkasan_rasio = pd.DataFrame({
    "perbandingan": [
        "SQL tanpa indeks / SQL berindeks",
        "SQL berindeks / key-value",
        "key-value agregasi / SQL GROUP BY",
        "SQLite agregasi / DuckDB agregasi",
        "SQL dua langkah / graph dua langkah",
        "NumPy spasial / STRtree spasial",
    ],
    "rasio": [
        rasio("cari 200 perjalanan - SQL tanpa indeks", "cari 200 perjalanan - SQL dengan indeks"),
        rasio("cari 200 perjalanan - SQL dengan indeks", "cari 200 perjalanan - key-value"),
        rasio("hitung per wilayah - key-value (buka semua nilai)", "hitung per wilayah - SQL GROUP BY"),
        rasio("SQLite - hitung per wilayah", "DuckDB - hitung per wilayah (langsung dari Parquet)"),
        rasio("2 langkah - SQL self-join (DuckDB)", "2 langkah - graph (NetworkX)"),
        rasio("cara 2 - NumPy, 1000 kotak", "cara 3 - STRtree, 1000 kotak"),
    ],
})
ringkasan_rasio["rasio"] = ringkasan_rasio["rasio"].round(2)
display(ringkasan_rasio)

,perbandingan,rasio
0,SQL tanpa indeks / SQL berindeks,1167.97
1,SQL berindeks / key-value,3.08
2,key-value agregasi / SQL GROUP BY,7.29
3,SQLite agregasi / DuckDB agregasi,15.16
4,SQL dua langkah / graph dua langkah,1.19
5,NumPy spasial / STRtree spasial,2.44


## O. Analisis Hasil

Jawaban berikut memakai angka dari hasil eksekusi notebook ini.

**1. Pencarian satu data.** Tabel tanpa indeks sekitar 1.170 kali lebih lambat dari tabel berindeks (4,3215 detik lawan 0,0037 detik untuk 200 pencarian), karena tanpa indeks setiap pencarian memeriksa baris satu per satu. Tabel berindeks sendiri sekitar 3,1 kali lebih lambat dari key-value (0,0037 detik lawan 0,0012 detik), karena pada key-value kunci langsung menunjuk letak data. Selisih yang benar-benar berarti hanya yang pertama. Pada Latihan 1 dengan 2.000 id, selisih keduanya cuma sekitar 14 mikrodetik per pencarian (20,24 lawan 6,62), yang tidak akan terasa oleh pengguna.

**2. Harga kesederhanaan.** Key-value sekitar 7,3 kali lebih lambat dari SQL GROUP BY untuk menghitung rata-rata per wilayah (1,4229 detik lawan 0,1951 detik). Sebabnya, setiap nilai di key-value hanya teks JSON yang harus dibuka dulu dengan json.loads, karena database tidak tahu isi nilainya.

**3. Data fleksibel.** Keuntungannya, dokumen ke-20.001 bisa membawa field cuaca tanpa ALTER TABLE, dan 4.094 dari 20.000 dokumen boleh tidak punya tip tanpa perlu kolom NULL. Risikonya, salah ketik nama field seperti biya akan diterima tanpa peringatan, jadi validasi menjadi tugas aplikasi.

**4. Penyimpanan per kolom.** DuckDB sekitar 15 kali lebih cepat dari SQLite (0,1139 detik lawan 1,7269 detik), dan tidak perlu tahap muat yang di SQLite memakan 9,1240 detik. Penyebab pertama adalah column pruning, karena query hanya membaca 2 dari 18 kolom. Penyebab kedua adalah pemrosesan banyak baris sekaligus dengan beberapa inti prosesor, yang tidak ada hubungannya dengan format kolom.

**5. Graph.** Untuk dua langkah, graph (0,0068 detik) dan SQL (0,0081 detik) memberi hasil yang sama, yaitu 214 zona, dengan selisih waktu hanya sekitar 1,3 milidetik. Hasil untuk k = 1 sampai 4 adalah 74, 214, 214, dan 214 zona. Graph tetap layak dipilih karena SQL membutuhkan satu JOIN tambahan untuk setiap langkah, sedangkan di graph cukup mengganti nilai k.

**6. Indeks spasial.** STRtree sekitar 9.500 kali lebih cepat dari perkiraan loop Python (0,2927 detik lawan sekitar 2.794 detik untuk 1.000 kotak), tetapi hanya sekitar 1,8 kali lebih cepat dari NumPy (0,5210 detik). Bedanya besar karena NumPy sudah bekerja secara vektor, sedangkan loop memeriksa titik satu per satu. Pada Latihan 5, untuk kotak 0,02 derajat STRtree malah lebih lambat dari NumPy (rasio 0,68), sedangkan untuk kotak 0,001 derajat ia sekitar 27 kali lebih cepat. Karena itu klaim percepatan harus selalu menyebut pembandingnya.

**7. Delta Lake.** Update tidak menghilangkan data lama, karena versi 1 masih terbaca utuh dengan 150.000 baris dan selisih total_bayar 20,0 sama dengan 20 baris yang dikoreksi. Penulisan yang gagal juga tidak meninggalkan jejak, karena penambahan kolom_asing ditolak dengan SchemaMismatchError dan nomor versi tetap 2. Setelah compaction, file di disk (24) masih lebih banyak daripada file aktif (1) karena file lama dibutuhkan untuk time travel dan baru dihapus secara fisik oleh VACUUM.

## P. Studi Kasus: Rancangan Penyimpanan Aplikasi Taksi

Angka pada tabel berikut diambil dari `benchmark_nosql.csv` dan output praktikum ini.

| Fitur | Cara menyimpan | Alasan berbasis hasil | Kelemahan |
|---|---|---|---|
| Detail perjalanan berdasarkan ID | Key-value, dengan tabel berindeks sebagai alternatif | 200 pencarian id selesai dalam 0,0012 detik pada key-value dan 0,0037 detik pada tabel berindeks, sedangkan tabel tanpa indeks butuh 4,3215 detik. Dua pilihan pertama sama-sama masih dalam hitungan milidetik. | Key-value lemah untuk menghitung: agregasi per wilayah 1,4229 detik, sekitar 7,3 kali lebih lambat dari SQL GROUP BY (0,1951 detik). |
| Dashboard pendapatan per wilayah per jam | Parquet + DuckDB | Agregasi per wilayah selesai dalam 0,1139 detik, sekitar 15 kali lebih cepat dari SQLite (1,7269 detik), dan filter satu wilayah hanya 0,0176 detik karena partition pruning. | Tidak cocok untuk sering mengubah atau menulis satu baris, dan isi dashboard baru berubah setelah data Parquet diperbarui. |
| Rekomendasi keterhubungan zona | Graph | Zona yang terjangkau dalam dua langkah dicari dalam 0,0068 detik dengan hasil sama seperti SQL (214 zona). Untuk k = 1 sampai 4 hasilnya 74, 214, 214, 214 zona, cukup dengan mengganti nilai k. | Graph harus dibangun ulang kalau data sumber berubah. Menghitung arus antar zonanya saja butuh 0,2170 detik. |
| Pencarian dalam radius 500 meter | Indeks spasial (STRtree) | 1.000 kotak selesai dalam 0,2927 detik termasuk membangun indeks, dibanding 0,5210 detik pada NumPy dan perkiraan sekitar 2.794 detik pada loop Python. Keuntungannya makin besar untuk kotak kecil (27,5 kali pada kotak 0,001 derajat). | Butuh koordinat yang valid. Data TLC hanya berisi ID zona, jadi angka K-6 berasal dari titik sintetis. |
| Audit dan pengembalian versi tarif | Delta Lake | Versi 0, 1, dan 2 tersimpan dan versi lama tetap terbaca (100.000 dan 150.000 baris). Koreksi 20 baris menghasilkan selisih tepat 20,0, dan restore(1) mengembalikan tabel ke 150.000 baris sebagai versi baru (versi 24). | Log transaksi dan file lama menumpuk: 24 file di disk padahal hanya 1 yang aktif, sehingga perlu compaction dan VACUUM dengan retention yang aman. |

### Data ganda dan sinkronisasi

Beberapa data memang terpaksa disimpan ganda. Key-value menyimpan salinan setiap perjalanan dalam bentuk JSON, graph menyimpan ringkasan arus antar zona, indeks spasial menyimpan titik lokasi, dan tabel Delta menyimpan salinan untuk audit. Semua salinan itu diturunkan dari satu sumber, yaitu trips_bersih, dan tidak pernah ditulis sendiri-sendiri. Kalau ada koreksi, perubahannya dilakukan di sumber (misalnya lewat update pada tabel Delta), lalu penyimpanan lain dibangun ulang dari situ.

Supaya salinannya tetap sama, jumlah baris dan hasil query dicek setiap kali penyimpanan dibangun ulang, seperti `hasil sama?` di K-2, K-4, dan K-5 serta jumlah key 300.000 di K-2. Nomor versi tabel Delta juga dicatat sebagai penanda pembaruan terakhir, jadi bisa diketahui salinan lain dibangun dari versi yang mana. Selama pembangunan ulang berjalan, salinan turunan bisa tertinggal sebentar. Untuk dashboard dan rekomendasi zona itu masih bisa diterima, tetapi untuk audit keuangan tidak, jadi audit selalu membaca langsung dari tabel Delta.

### Diagram aliran data

![Diagram aliran data](attachment:diagram_aliran_data.png)


# Q. Latihan

Latihan dikerjakan pada notebook praktikum yang sama setelah K-9.

## Q.1 Pencarian 2.000 ID

In [ ]:
random.seed(42)
id_uji_2000 = random.sample(range(1, len(data) + 1), 2_000)

def cari_sql_2000():
    cur = con.cursor()
    return [
        cur.execute(
            "SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)
        ).fetchone()
        for i in id_uji_2000
    ]

def cari_kv_2000():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji_2000]

t0 = time.perf_counter()
_ = cari_sql_2000()
waktu_sql_2000 = time.perf_counter() - t0

t0 = time.perf_counter()
_ = cari_kv_2000()
waktu_kv_2000 = time.perf_counter() - t0

print(f"SQL berindeks: {waktu_sql_2000:.6f} s | {waktu_sql_2000 / 2000 * 1e6:.2f} µs/pencarian")
print(f"Key-value    : {waktu_kv_2000:.6f} s | {waktu_kv_2000 / 2000 * 1e6:.2f} µs/pencarian")

SQL berindeks: 0.040489 s | 20.24 µs/pencarian
Key-value    : 0.013244 s | 6.62 µs/pencarian


## Q.2 Query document dan SQL

In [ ]:
zona_uji = int(sample.iloc[0]["zona_tujuan"])

jumlah_doc = len(tabel.search(
    (Q.tujuan.zona == zona_uji)
    & (Q.biaya.metode_bayar == "Tunai")
))

jumlah_sql = con.execute("""
    SELECT COUNT(*)
    FROM perjalanan
    WHERE zona_tujuan = ? AND metode_bayar = 'Tunai'
""", (zona_uji,)).fetchone()[0]

print("zona tujuan:", zona_uji)
print("document (20.000 baris):", jumlah_doc)
print("SQL (300.000 baris)     :", jumlah_sql)
print("sama?", jumlah_doc == jumlah_sql)
print("Catatan: cakupan data berbeda sehingga hasil jumlah dapat berbeda.")

zona tujuan: 164
document (20.000 baris): 93
SQL (300.000 baris)     : 1086
sama? False
Catatan: cakupan data berbeda sehingga hasil jumlah dapat berbeda.


## Q.3 SUM satu kolom dan beberapa kolom

In [ ]:
q_satu = "SELECT SUM(total_bayar) FROM perjalanan"
q_banyak = """
    SELECT SUM(total_bayar), SUM(jarak_mil),
           SUM(zona_jemput), SUM(zona_tujuan)
    FROM perjalanan
"""

t_satu = ukur("latihan: SUM satu kolom", lambda: con_d.sql(q_satu).fetchall())
t_banyak = ukur("latihan: SUM beberapa kolom", lambda: con_d.sql(q_banyak).fetchall())

print("hasil satu kolom:", t_satu)
print("hasil beberapa kolom:", t_banyak)

[latihan: SUM satu kolom] 0.0855 s
[latihan: SUM beberapa kolom] 0.1695 s
hasil satu kolom: [(81621442.45996779,)]
hasil beberapa kolom: [(81621442.45996639, 10231889.610000085, 497452755, 491330400)]


## Q.4 Out-degree graph dan SQL

In [ ]:
top_out_graph = sorted(
    G.out_degree(), key=lambda x: (-x[1], x[0])
)[:5]

top_out_sql = con_d.sql("""
    SELECT asal, COUNT(DISTINCT tujuan) AS jumlah_tujuan
    FROM arus_df
    GROUP BY asal
    ORDER BY jumlah_tujuan DESC, asal
    LIMIT 5
""").fetchall()

print("Graph:", top_out_graph)
print("SQL  :", top_out_sql)
print("hasil sama?", top_out_graph == top_out_sql)

Graph: [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
SQL  : [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
hasil sama? True


## Q.5 Ukuran kotak spasial

In [ ]:
def uji_sisi_spasial(setengah_sisi):
    kotak_uji = [
        box(x - setengah_sisi, y - setengah_sisi,
            x + setengah_sisi, y + setengah_sisi)
        for x, y in zip(cx, cy)
    ]

    t0 = time.perf_counter()
    hasil_numpy = []
    for x, y in zip(cx, cy):
        mask = (
            (bujur >= x - setengah_sisi)
            & (bujur <= x + setengah_sisi)
            & (lintang >= y - setengah_sisi)
            & (lintang <= y + setengah_sisi)
        )
        hasil_numpy.append(int(mask.sum()))
    waktu_numpy = time.perf_counter() - t0

    t0 = time.perf_counter()
    hasil_indeks = [
        len(pohon.query(k, predicate="contains")) for k in kotak_uji
    ]
    waktu_indeks = time.perf_counter() - t0

    return {
        "panjang_sisi_derajat": setengah_sisi * 2,
        "waktu_numpy": waktu_numpy,
        "waktu_strtree": waktu_indeks,
        "rasio_numpy_per_strtree": waktu_numpy / waktu_indeks,
        "hasil_sama": hasil_numpy == hasil_indeks,
    }

hasil_sisi = pd.DataFrame([
    uji_sisi_spasial(0.01),
    uji_sisi_spasial(0.0005),
])
display(hasil_sisi)

,panjang_sisi_derajat,waktu_numpy,waktu_strtree,rasio_numpy_per_strtree,hasil_sama
0,0.020,0.524885,0.774640,0.677586,True
1,0.001,0.526047,0.019157,27.459112,True


## Q.6 Time travel dan restore

In [ ]:
dt = DeltaTable(PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))

dt.restore(1)
dt = DeltaTable(PATH_DELTA)

print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


## Menyiapkan arsip Delta Lake

In [ ]:
PATH_ZIP_DELTA = f"{DIR_SIMPAN}/trips_delta"
hasil_zip = shutil.make_archive(PATH_ZIP_DELTA, "zip", root_dir=PATH_DELTA)
print("Arsip Delta tersimpan:", hasil_zip)

Arsip Delta tersimpan: /content/drive/MyDrive/BigData/Praktikum4/trips_delta.zip
